## Notebook 2: Landmark Extraction and Quality Check

This notebook combines two main steps from the project's data processing pipeline:

1.  **Landmark Extraction**: It processes raw images from the `dataset` folder, uses MediaPipe to detect hand landmarks, and saves the normalized landmark vectors into separate CSV files for each gesture class.
2.  **Quality Check**: After extraction, it inspects the generated CSV files for potential issues like missing values (NaNs), out-of-range data, or zero-variance features to ensure the dataset is clean and reliable for training.

### Part 1: Landmark Extraction

The following code is from `src/data/extract_landmarks.py`. It iterates through the class folders in the `dataset` directory, extracts hand landmarks from each image, preprocesses them into a normalized 42-dimension vector, and saves them to `model_artifacts/raw_landmarks`.

In [1]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm   
import warnings

warnings.filterwarnings('ignore')


class LandmarkExtractor:
    def __init__(self):
        # Requires mediapipe to be correctly installed/imported
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=True,
            max_num_hands=1,
            min_detection_confidence=0.7
        )
        self.mp_draw = mp.solutions.drawing_utils

    def preprocess_landmarks(self, landmarks, img_shape):
        """Convert 21 landmarks to 42‑dim normalized vector"""
        if not landmarks:
            return None

        h, w = img_shape[:2]
        pixel_landmarks = []
        for lm in landmarks:
            x_pixel = int(lm.x * w)
            y_pixel = int(lm.y * h)
            pixel_landmarks.append((x_pixel, y_pixel))

        base_x, base_y = pixel_landmarks[0]
        relative_landmarks = []
        for x, y in pixel_landmarks:
            relative_landmarks.append((x - base_x, y - base_y))

        vector = []
        for x_rel, y_rel in relative_landmarks:
            vector.extend([x_rel, y_rel])

        max_val = max(abs(v) for v in vector) if vector and max(abs(v) for v in vector) > 0 else 1.0
        normalized = [v / max_val for v in vector]

        return normalized

    def extract_from_folder(self, folder_path, class_id, output_csv):
        """Extract landmarks from all images in ../dataset/<class>/User_*"""
        folder = Path(folder_path)
        all_samples = []

        user_folders = sorted(folder.glob("User_*"))
        print(f"\n📁 Processing class '{folder.name}' (ID: {class_id})...")
        if not user_folders:
            print(f"❌ No User_* folders found in {folder}")
            return 0

        for user_folder in tqdm(user_folders, desc="Users"):
            user_samples = []
            img_files = list(user_folder.glob("*.jpg")) + list(user_folder.glob("*.png"))

            for img_file in tqdm(img_files, desc=f"{user_folder.name}", leave=False):
                try:
                    image = cv2.imread(str(img_file))
                    if image is None:
                        continue

                    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                    results = self.hands.process(image_rgb)

                    if results.multi_hand_landmarks and results.multi_hand_landmarks[0]:
                        landmarks = results.multi_hand_landmarks[0].landmark
                        vector = self.preprocess_landmarks(landmarks, image.shape)

                        if vector is not None:
                            user_samples.append([class_id] + vector)

                except Exception as e:
                    print(f"Error processing {img_file}: {e}")
                    continue

            all_samples.extend(user_samples)
            print(f"  {user_folder.name}: {len(user_samples)} valid samples")

        if all_samples:
            df = pd.DataFrame(all_samples, columns=['class'] + [f'feature_{i}' for i in range(42)])
            df.to_csv(output_csv, index=False)
            print(f"✅ Saved {len(all_samples)} samples to {output_csv}")
            return len(all_samples)
        else:
            print(f"❌ No valid samples found for {folder.name}")
            return 0


def main_extraction():
    # Class mapping (must match keypoint_classifier_label.csv)
    class_mapping = {
        'afraid': 0,
        'agree': 1,
        'assistance': 2,
        'bad': 3,
        'become': 4,
        'college': 5,
        'doctor': 6,
        'from': 7,
        'pain': 8,
        'pray': 9,
        'secondary': 10,
        'skin': 11,
        'small': 12,
        'specific': 13,
        'stand': 14,
        'today': 15,
        'warn': 16,
        'which': 17,
        'work': 18,
        'you': 19
    }

    extractor = LandmarkExtractor()
    total_samples = 0

    # Output directory for the raw landmark CSVs
    output_root = Path("../model_artifacts/raw_landmarks")
    os.makedirs(output_root, exist_ok=True)

    # Root directory of the dataset
    dataset_root = Path("../dataset")

    for class_name, class_id in class_mapping.items():
        class_folder = dataset_root / class_name
        output_csv = output_root / f"{class_name}.csv"

        if not class_folder.exists():
            print(f"❌ Missing folder: {class_folder}")
            continue

        samples = extractor.extract_from_folder(class_folder, class_id, output_csv)
        total_samples += samples

    print(f"\n🎉 EXTRACTION COMPLETE!")
    print(f"📊 Total landmark vectors extracted: {total_samples}")
    print(f"📁 Files saved in: {output_root}")

    print("\n📋 SUMMARY:")
    for class_name in class_mapping:
        csv_file = output_root / f"{class_name}.csv"
        if csv_file.exists():
            df = pd.read_csv(csv_file)
            print(f"  {class_name}: {len(df)} samples ✓")
        else:
            print(f"  {class_name}: 0 samples ❌")

# Run the landmark extraction process
main_extraction()


📁 Processing class 'afraid' (ID: 0)...


Users:  17%|█▋        | 1/6 [00:08<00:42,  8.47s/it]

  User_1: 150 valid samples


Users:  33%|███▎      | 2/6 [00:18<00:36,  9.24s/it]

  User_2: 150 valid samples


...

  User_4: 118 valid samples


Users:  83%|████████▎ | 5/6 [01:05<00:13, 13.11s/it]

  User_5: 98 valid samples


Users: 100%|██████████| 6/6 [01:18<00:00, 13.13s/it]


  User_6: 108 valid samples
✅ Saved 713 samples to ..\model_artifacts\raw_landmarks\you.csv

🎉 EXTRACTION COMPLETE!
📊 Total landmark vectors extracted: 14886
📁 Files saved in: ..\model_artifacts\raw_landmarks

📋 SUMMARY:
  afraid: 832 samples ✓
  agree: 846 samples ✓
  assistance: 561 samples ✓
  bad: 859 samples ✓
  become: 732 samples ✓
  college: 814 samples ✓
  doctor: 442 samples ✓
  from: 668 samples ✓
  pain: 733 samples ✓
  pray: 341 samples ✓
  secondary: 863 samples ✓
  skin: 847 samples ✓
  small: 625 samples ✓
  specific: 797 samples ✓
  stand: 741 samples ✓
  today: 844 samples ✓
  warn: 873 samples ✓
  which: 868 samples ✓
  work: 887 samples ✓
  you: 713 samples ✓


### Part 2: Landmark Quality Check

This section corresponds to `src/inference/check_landmarks_quality.py`. It runs a series of checks on the CSV files generated in the previous step to ensure they are suitable for training.

In [2]:
import pandas as pd
from pathlib import Path

def check_quality():
    # Directory where the raw landmark CSVs are stored
    raw_landmarks_dir = Path("../model_artifacts/raw_landmarks")
    total_samples = 0
    valid_samples = 0

    print("🔍 LANDMARK QUALITY CHECK")
    print("=" * 50)
    print(f"Looking for CSVs in: {raw_landmarks_dir.resolve()}\n")

    if not raw_landmarks_dir.exists():
        print(f"❌ Folder not found: {raw_landmarks_dir}")
        return

    csv_files = list(raw_landmarks_dir.glob("*.csv"))
    if not csv_files:
        print("❌ No CSV files found in raw_landmarks/. Run landmark extraction first.")
        return

    for csv_file in sorted(csv_files):
        df = pd.read_csv(csv_file)
        samples_count = len(df)
        total_samples += samples_count

        # Check for NaN values
        nan_count = df.isnull().sum().sum()

        # Check for invalid ranges (landmarks normalized ~[-1,1], allowing some margin)
        invalid_range = ((df.iloc[:, 1:] < -2) | (df.iloc[:, 1:] > 2)).sum().sum()

        # Check for features with zero variance
        zero_var = (df.iloc[:, 1:].var() == 0).sum()

        # A sample is valid if it has no NaNs, is within the expected range, and doesn't have zero variance
        valid_count = samples_count - nan_count - invalid_range - zero_var
        valid_samples += valid_count

        status = "✅" if valid_count == samples_count else "⚠️"
        print(
            f"{status} {csv_file.name}: {samples_count} total, {valid_count} valid "
            f"(NaNs: {nan_count}, Invalid Range: {invalid_range}, Zero Variance: {zero_var})"
        )

    print("\n" + "=" * 50)
    print(f"🎯 TOTAL: {valid_samples}/{total_samples} valid samples")
    ready_for_next_phase = total_samples > 0 and valid_samples == total_samples
    print(f"✅ Ready for next phase (merging): {ready_for_next_phase}")

# Run the quality check
check_quality()

🔍 LANDMARK QUALITY CHECK
Looking for CSVs in: D:\Final year project-landmarks\model_artifacts\raw_landmarks

⚠️ afraid.csv: 832 total, 830 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ agree.csv: 846 total, 844 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ assistance.csv: 561 total, 559 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ bad.csv: 859 total, 857 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ become.csv: 732 total, 730 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ college.csv: 814 total, 812 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ doctor.csv: 442 total, 440 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ from.csv: 668 total, 666 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ pain.csv: 733 total, 731 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ pray.csv: 341 total, 339 valid (NaNs: 0, Invalid Range: 0, Zero Variance: 2)
⚠️ secondary.csv: 863 total, 861 valid (NaNs: 0, Invalid Range: 0, Zero Vari